# Bus_Himmelev

In [7]:
navne_med_farver = [
    {"navn": "Anna", "farve": "Rød"},
    {"navn": "Benjamin", "farve": "Blå"},
    {"navn": "Clara", "farve": "Grøn"},
    {"navn": "Daniel", "farve": "Grå"},
    {"navn": "Emma", "farve": "Lilla"},
    {"navn": "Bus 201", "farve": "Gul"},
]

for person in navne_med_farver:
    print(f"{person['navn']}: {person['farve']}")

Anna: Rød
Benjamin: Blå
Clara: Grøn
Daniel: Grå
Emma: Lilla
Bus 201: Gul


In [8]:
from simulated_city.maplibre_live import LiveMapLibreMap
import random
import time
from IPython.display import display

# Givet som (lat, lng)
LAT = 55.65777549836654
LNG = 12.105861370948363

# anymap_ts bruger (lng, lat)
m = LiveMapLibreMap(center=(LNG, LAT), zoom=18, pitch=58, bearing=25, height="600px", width="100%")
m.add_basemap("OpenStreetMap.Mapnik")
m.add_3d_buildings()
m.add_marker(LNG, LAT, name="punkt", color="#1E88E5", popup="Valgt koordinat")

random.seed()
personer = [p for p in navne_med_farver if not p["navn"].lower().startswith("bus")]

# Hjørner (h) for området personer må bevæge sig indenfor (input: lat, lng -> [lng, lat])
h1 = [12.106759008261267, 55.65861989708783]
h2 = [12.106816349671297, 55.657810931391566]
h3 = [12.104950909565892, 55.65782359387148]
h4 = [12.105068108217221, 55.65857206333853]
human_area = [h1, h2, h3, h4]

# Bus 201-rute (input er lat/lng -> konverteret til [lng, lat])
p1 = [12.107061036174597, 55.65773066383307]
p2 = [12.105830262981327, 55.65777189215283]
p3 = [12.10402256017052, 55.657776213658096]
p4 = [12.104235395707752, 55.659645821205125]
p5 = [12.107563231968914, 55.65970153877991]
p6 = [12.107958394657734, 55.658674908889644]
p7 = [12.107126427302157, 55.65773627741259]

# Hastighedsoptimering: spring direkte fra p4 til p5 (kan sættes til False)
TELEPORT_P4_P5 = True

# Tegn kun hvis markøren faktisk flytter sig synligt
_DRAW_EPS = 1e-7
_last_drawn = {}

def _should_redraw(marker_id, lng, lat, on_bus_flag=None):
    prev = _last_drawn.get(marker_id)
    if prev is None:
        _last_drawn[marker_id] = (lng, lat, on_bus_flag)
        return True
    prev_lng, prev_lat, prev_flag = prev
    moved = (
        abs(prev_lng - lng) > _DRAW_EPS
        or abs(prev_lat - lat) > _DRAW_EPS
        or prev_flag != on_bus_flag
    )
    if moved:
        _last_drawn[marker_id] = (lng, lat, on_bus_flag)
    return moved

def point_in_polygon(lng, lat, polygon):
    inside = False
    j = len(polygon) - 1
    for i in range(len(polygon)):
        xi, yi = polygon[i]
        xj, yj = polygon[j]
        intersects = ((yi > lat) != (yj > lat)) and (
            lng < (xj - xi) * (lat - yi) / ((yj - yi) + 1e-12) + xi
        )
        if intersects:
            inside = not inside
        j = i
    return inside

def random_point_in_polygon(polygon):
    lng_values = [p[0] for p in polygon]
    lat_values = [p[1] for p in polygon]
    min_lng, max_lng = min(lng_values), max(lng_values)
    min_lat, max_lat = min(lat_values), max(lat_values)
    while True:
        candidate_lng = random.uniform(min_lng, max_lng)
        candidate_lat = random.uniform(min_lat, max_lat)
        if point_in_polygon(candidate_lng, candidate_lat, polygon):
            return [candidate_lng, candidate_lat]

def person_color(on_bus):
    return "rgba(67,160,71,0.45)" if on_bus else "#43A047"

def move_person_marker(marker_id, lng, lat, navn, farve, on_bus=False):
    if not _should_redraw(marker_id, lng, lat, on_bus):
        return
    m.move_marker(
        marker_id,
        (lng, lat),
        color=person_color(on_bus),
        popup=f"{navn} ({farve})",
    )

# Opret én markør pr. person og gem position
person_positions = {}
for person in personer:
    marker_id = f"person-{person['navn']}"
    lng, lat = random_point_in_polygon(human_area)
    person_positions[marker_id] = {
        "lng": lng,
        "lat": lat,
        "navn": person["navn"],
        "farve": person["farve"],
        "on_bus": False,
        "status": "free",
        "unboard_target": None,
        "unboard_progress": 0.0,
        "unboard_step": 0.08,
    }
    move_person_marker(marker_id, lng, lat, person["navn"], person["farve"], on_bus=False)

def tick_people(step_size=0.00003, locked_ids=None):
    if locked_ids is None:
        locked_ids = set()

    for marker_id, info in person_positions.items():
        if marker_id in locked_ids:
            continue

        # Folk, der er ved at stige af, bevæger sig mod deres mål uden at stoppe bussen
        if info["status"] == "unboarding":
            t = min(1.0, info["unboard_progress"] + info["unboard_step"] )
            target_lng, target_lat = info["unboard_target"]
            start_lng, start_lat = info["unboard_start"]
            next_lng = start_lng + (target_lng - start_lng) * t
            next_lat = start_lat + (target_lat - start_lat) * t
            info["lng"] = next_lng
            info["lat"] = next_lat
            info["unboard_progress"] = t
            move_person_marker(marker_id, next_lng, next_lat, info["navn"], info["farve"], on_bus=False)
            if t >= 1.0:
                info["status"] = "free"
                info["on_bus"] = False
                info["unboard_target"] = None
            continue

        if info["on_bus"] or info["status"] != "free":
            continue

        current_lng = info["lng"]
        current_lat = info["lat"]
        next_lng = current_lng + random.uniform(-step_size, step_size)
        next_lat = current_lat + random.uniform(-step_size, step_size)

        for _ in range(6):
            if point_in_polygon(next_lng, next_lat, human_area):
                break
            next_lng = current_lng + random.uniform(-step_size, step_size)
            next_lat = current_lat + random.uniform(-step_size, step_size)
        else:
            next_lng, next_lat = random_point_in_polygon(human_area)

        info["lng"] = next_lng
        info["lat"] = next_lat
        move_person_marker(marker_id, next_lng, next_lat, info["navn"], info["farve"], on_bus=False)

# Opret én bus-markør, som flyttes rundt (ingen efterladte punkter)
bus_position = {"lng": p1[0], "lat": p1[1]}
m.move_marker("bus-201", (p1[0], p1[1]), color="#FDD835", popup="Bus 201")
_last_drawn["bus-201"] = (p1[0], p1[1], None)

def move_bus(lng, lat):
    bus_position["lng"] = lng
    bus_position["lat"] = lat

    if _should_redraw("bus-201", lng, lat, None):
        m.move_marker("bus-201", (lng, lat), color="#FDD835", popup="Bus 201")

    # Ombord-personer følger bussen 1:1
    for marker_id, info in person_positions.items():
        if not info["on_bus"]:
            continue
        info["lng"] = lng
        info["lat"] = lat
        move_person_marker(marker_id, lng, lat, info["navn"], info["farve"], on_bus=True)

def bus_is_at_stop(stop_point, tolerance=0.00001):
    stop_lng, stop_lat = stop_point
    return (
        abs(bus_position["lng"] - stop_lng) <= tolerance
        and abs(bus_position["lat"] - stop_lat) <= tolerance
    )

def count_onboard():
    return sum(1 for info in person_positions.values() if info["on_bus"] )

def board_people_at_p2(max_board=3, board_seconds=2.0, board_steps=20):
    candidates = [mid for mid, info in person_positions.items() if not info["on_bus"] and info["status"] == "free"]
    if not candidates:
        return 0

    selected = random.sample(candidates, k=min(max_board, len(candidates)))
    starts = {mid: (person_positions[mid]["lng"], person_positions[mid]["lat"]) for mid in selected}

    # Led de valgte personer hen til bus-positionen med stabil interpolation
    for step in range(1, board_steps + 1):
        t = step / board_steps
        target_lng = bus_position["lng"]
        target_lat = bus_position["lat"]

        for marker_id in selected:
            info = person_positions[marker_id]
            start_lng, start_lat = starts[marker_id]
            next_lng = start_lng + (target_lng - start_lng) * t
            next_lat = start_lat + (target_lat - start_lat) * t
            info["lng"] = next_lng
            info["lat"] = next_lat
            move_person_marker(marker_id, next_lng, next_lat, info["navn"], info["farve"], on_bus=False)

        tick_people(locked_ids=set(selected))
        time.sleep(board_seconds / board_steps)

    for marker_id in selected:
        info = person_positions[marker_id]
        info["on_bus"] = True
        info["status"] = "on_bus"
        info["lng"] = bus_position["lng"]
        info["lat"] = bus_position["lat"]
        move_person_marker(marker_id, bus_position["lng"], bus_position["lat"], info["navn"], info["farve"], on_bus=True)

    return len(selected)

def start_unboard_at_p4():
    riders = [mid for mid, info in person_positions.items() if info["on_bus"] ]
    if not riders:
        return

    for marker_id in riders:
        info = person_positions[marker_id]
        target = random_point_in_polygon(human_area)
        info["status"] = "unboarding"
        info["on_bus"] = False
        info["unboard_target"] = target
        info["unboard_start"] = (bus_position["lng"], bus_position["lat"] )
        info["unboard_progress"] = 0.0

def move_segment(start, end, seconds=6, steps=60):
    frame_dt = seconds / steps
    next_ts = time.perf_counter()

    for i in range(1, steps + 1):
        t = i / steps
        lng = start[0] + (end[0] - start[0]) * t
        lat = start[1] + (end[1] - start[1]) * t
        move_bus(lng, lat)
        tick_people()

        next_ts += frame_dt
        sleep_s = next_ts - time.perf_counter()
        if sleep_s > 0:
            time.sleep(sleep_s)

def teleport_bus(target):
    move_bus(target[0], target[1])
    tick_people()

def pause_at_p2_with_boarding(board_target=3, max_wait=8.0, interval=0.1):
    # Sensor: kør videre så snart alle ny-boardede passagerer er ombord
    if not bus_is_at_stop(p2):
        return

    onboard_before = count_onboard()
    boarded_now = board_people_at_p2(max_board=board_target, board_seconds=2.0, board_steps=20)
    if boarded_now <= 0:
        return

    target_onboard = onboard_before + boarded_now
    deadline = time.perf_counter() + max_wait

    while count_onboard() < target_onboard and time.perf_counter() < deadline:
        tick_people()
        time.sleep(interval)

display(m)

# Uendeligt loop: p1 -> p2 (boarding) -> p3 -> p4 (afstigning) -> p5 -> p6 -> p7 -> p1
while True:
    move_segment(p1, p2, seconds=6, steps=60)
    pause_at_p2_with_boarding(board_target=3, max_wait=8.0, interval=0.1)
    move_segment(p2, p3, seconds=6, steps=60)
    move_segment(p3, p4, seconds=6, steps=60)
    if bus_is_at_stop(p4):
        start_unboard_at_p4()

    if TELEPORT_P4_P5:
        teleport_bus(p5)
    else:
        move_segment(p4, p5, seconds=6, steps=60)

    # Hurtigere bus på denne del af ruten
    move_segment(p5, p6, seconds=3.0, steps=45)
    move_segment(p6, p7, seconds=3.0, steps=45)
    move_segment(p7, p1, seconds=2.0, steps=28)
    # Ingen pause ved p1 - næste tur starter med det samme

KeyboardInterrupt: 

In [4]:
# State-machine visning for Bus 201 + passagerer
import pandas as pd
from datetime import datetime

required_points = ["p1", "p2", "p3", "p4", "p5", "p6", "p7"]
missing = [name for name in required_points if name not in globals()]
if missing:
    raise RuntimeError(f"Kør bus-cellen først. Mangler variabler: {missing}")

# Hent person-data hvis den findes fra bus-cellen
if "person_positions" in globals():
    initial_free = sum(1 for info in person_positions.values() if info.get("status") == "free")
    initial_on_bus = sum(1 for info in person_positions.values() if info.get("on_bus"))
else:
    # Fallback hvis bus-cellen ikke har været kørt
    initial_free = 5
    initial_on_bus = 0

sim = {
    "bus_state": "ON_ROUTE",
    "people_free": initial_free,
    "people_on_bus": initial_on_bus,
    "people_unboarding": 0,
}

state_counters = {
    "ON_ROUTE": 0,
    "AT_STOP": 0,
    "BOARDING": 0,
    "UNBOARDING": 0,
}

transitions = {}
event_rows = []

def add_transition(from_state, to_state):
    key = f"{from_state} -> {to_state}"
    transitions[key] = transitions.get(key, 0) + 1

def log_event(bus_position_label, event_label):
    state_counters[sim["bus_state"]] += 1
    event_rows.append(
        {
            "time": datetime.now().strftime("%H:%M:%S"),
            "bus_position": bus_position_label,
            "bus_state": sim["bus_state"],
            "event": event_label,
            "people_free": sim["people_free"],
            "people_on_bus": sim["people_on_bus"],
            "people_unboarding": sim["people_unboarding"],
        }
    )

def set_bus_state(new_state):
    old_state = sim["bus_state"]
    if old_state != new_state:
        add_transition(old_state, new_state)
        sim["bus_state"] = new_state

# Simulér én fuld runde af state-machine
route = ["p1", "p2", "p3", "p4", "p5", "p6", "p7", "p1"]

for i in range(len(route) - 1):
    from_p = route[i]
    to_p = route[i + 1]

    set_bus_state("ON_ROUTE")
    log_event(from_p, f"Afgang fra {from_p} mod {to_p}")

    # Stop ved p2: boarding
    if to_p == "p2":
        set_bus_state("AT_STOP")
        log_event("p2", "Ankomst til stop p2")

        set_bus_state("BOARDING")
        board_target = min(3, sim["people_free"])
        sim["people_free"] -= board_target
        sim["people_on_bus"] += board_target
        log_event("p2", f"Boarding færdig ({board_target} passagerer)")

    # Stop ved p4: unboarding
    if to_p == "p4":
        set_bus_state("AT_STOP")
        log_event("p4", "Ankomst til stop p4")

        set_bus_state("UNBOARDING")
        unboard_now = sim["people_on_bus"]
        sim["people_on_bus"] = 0
        sim["people_unboarding"] = unboard_now
        log_event("p4", f"Afstigning startet ({unboard_now} passagerer)")

        # Når de er i firkanten igen
        sim["people_unboarding"] = 0
        sim["people_free"] += unboard_now
        log_event("h-område", f"Afstigning færdig ({unboard_now} tilbage i området)")

# Resultater
events_df = pd.DataFrame(event_rows)
transitions_df = pd.DataFrame(
    [{"transition": k, "count": v} for k, v in transitions.items()]
).sort_values("count", ascending=False)

print("STATE-MACHINE OVERBLIK (1 runde)")
print("=" * 50)
print("State-counters:")
for state_name, count in state_counters.items():
    print(f"- {state_name}: {count}")

print("\nTransitions:")
for _, row in transitions_df.iterrows():
    print(f"- {row['transition']}: {int(row['count'])}")

print("\nEvent-log:")
display(events_df)

STATE-MACHINE OVERBLIK (1 runde)
State-counters:
- ON_ROUTE: 7
- AT_STOP: 2
- BOARDING: 1
- UNBOARDING: 2

Transitions:
- ON_ROUTE -> AT_STOP: 2
- AT_STOP -> BOARDING: 1
- BOARDING -> ON_ROUTE: 1
- AT_STOP -> UNBOARDING: 1
- UNBOARDING -> ON_ROUTE: 1

Event-log:


,time,bus_position,bus_state,event,people_free,people_on_bus,people_unboarding
0,12:35:17,p1,ON_ROUTE,Afgang fra p1 mod p2,5,0,0
1,12:35:17,p2,AT_STOP,Ankomst til stop p2,5,0,0
2,12:35:17,p2,BOARDING,Boarding færdig (3 passagerer),2,3,0
3,12:35:17,p2,ON_ROUTE,Afgang fra p2 mod p3,2,3,0
4,12:35:17,p3,ON_ROUTE,Afgang fra p3 mod p4,2,3,0
5,12:35:17,p4,AT_STOP,Ankomst til stop p4,2,3,0
6,12:35:17,p4,UNBOARDING,Afstigning startet (3 passagerer),2,0,3
7,12:35:17,h-område,UNBOARDING,Afstigning færdig (3 tilbage i området),5,0,0
8,12:35:17,p4,ON_ROUTE,Afgang fra p4 mod p5,5,0,0
9,12:35:17,p5,ON_ROUTE,Afgang fra p5 mod p6,5,0,0


In [7]:
# Live state-machine visning (realtime)
import time
from datetime import datetime
from IPython.display import clear_output, display
import pandas as pd

def run_live_state_machine(cycles=1, tick_seconds=0.8):
    sim = {
        "bus_state": "ON_ROUTE",
        "people_free": 5,
        "people_on_bus": 0,
        "people_unboarding": 0,
    }

    state_counters = {
        "ON_ROUTE": 0,
        "AT_STOP": 0,
        "BOARDING": 0,
        "UNBOARDING": 0,
    }

    transitions = {}
    event_rows = []

    def add_transition(from_state, to_state):
        key = f"{from_state} -> {to_state}"
        transitions[key] = transitions.get(key, 0) + 1

    def set_bus_state(new_state):
        old_state = sim["bus_state"]
        if old_state != new_state:
            add_transition(old_state, new_state)
            sim["bus_state"] = new_state

    def push_event(position_label, event_label):
        state_counters[sim["bus_state"]] += 1
        event_rows.append(
            {
                "time": datetime.now().strftime("%H:%M:%S"),
                "position": position_label,
                "bus_state": sim["bus_state"],
                "event": event_label,
                "free": sim["people_free"],
                "on_bus": sim["people_on_bus"],
                "unboarding": sim["people_unboarding"],
            }
        )

        clear_output(wait=True)
        print("LIVE STATE-MACHINE (Bus 201)")
        print("=" * 46)
        print(f"Nuværende state: {sim['bus_state']}")
        print(f"Passagerer -> fri: {sim['people_free']}, ombord: {sim['people_on_bus']}, afstiger: {sim['people_unboarding']}")
        print("\nState-counters:")
        for name, count in state_counters.items():
            print(f"- {name}: {count}")

        if transitions:
            trans_df = pd.DataFrame(
                [{"transition": k, "count": v} for k, v in transitions.items()]
            ).sort_values("count", ascending=False)
            print("\nTransitions:")
            display(trans_df)

        print("\nSeneste events:")
        display(pd.DataFrame(event_rows).tail(10))
        time.sleep(tick_seconds)

    route = ["p1", "p2", "p3", "p4", "p5", "p6", "p7", "p1"]

    for _ in range(cycles):
        for i in range(len(route) - 1):
            from_p = route[i]
            to_p = route[i + 1]

            set_bus_state("ON_ROUTE")
            push_event(from_p, f"Afgang fra {from_p} mod {to_p}")

            if to_p == "p2":
                set_bus_state("AT_STOP")
                push_event("p2", "Ankomst til stop p2")

                set_bus_state("BOARDING")
                board_now = min(3, sim["people_free"])
                sim["people_free"] -= board_now
                sim["people_on_bus"] += board_now
                push_event("p2", f"Boarding færdig ({board_now} passagerer)")

            if to_p == "p4":
                set_bus_state("AT_STOP")
                push_event("p4", "Ankomst til stop p4")

                set_bus_state("UNBOARDING")
                unboard_now = sim["people_on_bus"]
                sim["people_on_bus"] = 0
                sim["people_unboarding"] = unboard_now
                push_event("p4", f"Afstigning startet ({unboard_now} passagerer)")

                sim["people_unboarding"] = 0
                sim["people_free"] += unboard_now
                push_event("h-område", f"Afstigning færdig ({unboard_now} tilbage i området)")

    print("\nLive-afspilning færdig.")

# Kør 2 runder live. Justér evt. tick_seconds for hurtigere/langsommere afspilning.
run_live_state_machine(cycles=2, tick_seconds=4.0)

LIVE STATE-MACHINE (Bus 201)
Nuværende state: ON_ROUTE
Passagerer -> fri: 2, ombord: 3, afstiger: 0

State-counters:
- ON_ROUTE: 2
- AT_STOP: 1
- BOARDING: 1
- UNBOARDING: 0

Transitions:


,transition,count
0,ON_ROUTE -> AT_STOP,1
1,AT_STOP -> BOARDING,1
2,BOARDING -> ON_ROUTE,1



Seneste events:


,time,position,bus_state,event,free,on_bus,unboarding
0,12:36:56,p1,ON_ROUTE,Afgang fra p1 mod p2,5,0,0
1,12:37:00,p2,AT_STOP,Ankomst til stop p2,5,0,0
2,12:37:04,p2,BOARDING,Boarding færdig (3 passagerer),2,3,0
3,12:37:08,p2,ON_ROUTE,Afgang fra p2 mod p3,2,3,0


KeyboardInterrupt: 